In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix, recall_score
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [5]:
df = pd.read_csv('DB_LIMPIA.csv', sep=";")
df.head(5)

,Unnamed: 0,VIC_SEXO,VIC_EDAD,TOTAL_HIJOS,VIC_ESCOLARIDAD,VIC_EST_CIV,VIC_GRUPET,VIC_TRABAJA,VIC_DISC,VIC_REL_AGR,...,HEC_TIPAGRE,INST_DONDE_DENUNCIO,AGR_SEXO,AGR_EDAD,AGR_ESCOLARIDAD,AGR_EST_CIV,AGR_GRUPET,AGR_TRABAJA,INST_DENUN_HECHO,MEDIDAS_SEGURIDAD
0,0,Mujeres,11,-1,Primaria,Desconocido,Ladino,No,Desconocido,Hijos(as),...,Física-sexual,NaN,Hombres,58,Primaria,Soltero,Ladino,No,Ministerio Público,Desconocido
1,1,Hombres,4,-1,Desconocido,Desconocido,Ladino,Desconocido,No,Hijos(as),...,Física-psicológica,NaN,Mujeres,27,Primaria,Soltero,Ladino,Si,Procuraduría de los Derechos Humanos,Desconocido
2,2,Hombres,11,-1,Primaria,Desconocido,Ladino,No,No,Hijos(as),...,Física-psicológica,NaN,Hombres,35,Ninguna,Unido,Ladino,Si,Procuraduría de los Derechos Humanos,Desconocido
3,3,Mujeres,6,-1,Desconocido,Desconocido,Ladino,Desconocido,No,Hijos(as),...,Psicológica,NaN,Hombres,35,Primaria,Soltero,Ladino,Si,Organismo Judicial,Si
4,4,Hombres,11,-1,Primaria,Desconocido,Ladino,No,No,Hijos(as),...,Psicológica,NaN,Hombres,35,Primaria,Soltero,Ladino,Si,Organismo Judicial,Si


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 358998 entries, 0 to 358997
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   Unnamed: 0           358998 non-null  int64
 1   VIC_SEXO             358998 non-null  str  
 2   VIC_EDAD             358998 non-null  int64
 3   TOTAL_HIJOS          358998 non-null  int64
 4   VIC_ESCOLARIDAD      358998 non-null  str  
 5   VIC_EST_CIV          358998 non-null  str  
 6   VIC_GRUPET           358998 non-null  str  
 7   VIC_TRABAJA          358998 non-null  str  
 8   VIC_DISC             358998 non-null  str  
 9   VIC_REL_AGR          358998 non-null  str  
 10  OTRAS_VICTIMAS       358998 non-null  int64
 11  HEC_DIA              358998 non-null  int64
 12  HEC_MES              358998 non-null  str  
 13  HEC_ANO              358998 non-null  int64
 14  HEC_AREA             358998 non-null  str  
 15  HEC_TIPAGRE          358998 non-null  str  
 16  INST_DONDE_DE

Se procede a crear un conjunto de datos en el que se encuentren etiquetados en VICTIMAS y AGRESORES, como se ha hecho en el primer proyecto.

In [10]:
# Se crea el DataFrame de víctimas.
df_victimas = df.loc[:, ['VIC_SEXO', 'VIC_EDAD', 'VIC_ESCOLARIDAD', 'VIC_EST_CIV', 'VIC_GRUPET', 'VIC_TRABAJA']]
df_victimas.columns = ['SEXO', 'EDAD', 'ESCOLARIDAD', 'ESTADO_CIVIL', 'GRUPO_ETNICO', 'TRABAJA']
df_victimas['ETIQUETA'] = 'VICTIMA'

# Se crea el DataFrame de agresores.
df_agresores = df.loc[:, ['AGR_SEXO', 'AGR_EDAD', 'AGR_ESCOLARIDAD', 'AGR_EST_CIV', 'AGR_GRUPET', 'AGR_TRABAJA']]
df_agresores.columns = ['SEXO', 'EDAD', 'ESCOLARIDAD', 'ESTADO_CIVIL', 'GRUPO_ETNICO', 'TRABAJA']
df_agresores['ETIQUETA'] = 'AGRESOR'

# Se crea el DataFrame de ambos conjuntos etiquetados.
DF = pd.concat([df_victimas, df_agresores])
DF.head(10)

,SEXO,EDAD,ESCOLARIDAD,ESTADO_CIVIL,GRUPO_ETNICO,TRABAJA,ETIQUETA
0,Mujeres,11,Primaria,Desconocido,Ladino,No,VICTIMA
1,Hombres,4,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
2,Hombres,11,Primaria,Desconocido,Ladino,No,VICTIMA
3,Mujeres,6,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
4,Hombres,11,Primaria,Desconocido,Ladino,No,VICTIMA
5,Hombres,1,Desconocido,Desconocido,Ladino,Desconocido,VICTIMA
6,Mujeres,10,Ninguna,Desconocido,Ladino,No,VICTIMA
7,Hombres,8,Ninguna,Desconocido,Desconocido,Desconocido,VICTIMA
8,Mujeres,10,Primaria,Desconocido,Ladino,Desconocido,VICTIMA
9,Mujeres,10,Primaria,Desconocido,Ladino,Si,VICTIMA


Se presentan ahora las estadísticas del objetivo:

In [12]:
DF['ETIQUETA'].describe()

count      717996
unique          2
top       VICTIMA
freq       358998
Name: ETIQUETA, dtype: object

Tratamiento de N.A.

In [14]:
df = DF.fillna(DF.median(numeric_only=True))#Tratamiento de NA

In [16]:
#Se procesan las etiquetas, al retirarlas del dataframe
categoria=[]
for x in df["ETIQUETA"]:
    categoria.append(x)
df = df.drop(columns=['ETIQUETA'])

Dado que hay varias variables tipo string, se procederá ha realizar una codificación tanto entera como binaria, según sea el caso, de estas variables

In [20]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("bin", "passthrough", binary_cols),  # ya están en 0/1
    ("cat", OneHotEncoder(), multi_cols)
])

NameError: name 'num_cols' is not defined

SEPARACIÓN DE DATOS

In [17]:
X_train, X_test, y_train, y_test = train_test_split(df, categoria, test_size=0.2, random_state=42, stratify=categoria)

In [18]:
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("svm", SVC())
])

pipe.fit(X_train, y_train)

ValueError: could not convert string to float: 'Hombres'